In [ ]:
!pip install -q langgraph langchain-openai langchain-core python-dotenv

In [ ]:
import os
from typing import TypedDict, List, Literal
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, START, END

# Set your OpenAI API key (or load from environment/.env)
os.environ["OPENAI_API_KEY"] = "your-openai-api-key-here"

# Initialize the LLM
llm = ChatOpenAI(model="gpt-4o", temperature=0.2)
print("Environment and LLM initialized successfully!")

In [ ]:
class MeetingState(TypedDict):
    transcript: str
    topics: List[str]
    summary: str
    action_items: List[str]
    has_action_items: bool
    priority: str
    final_report: str

In [ ]:
def topic_extraction_node(state: MeetingState):
    print("--- Running Topic Extraction Agent ---")
    prompt = f"""
    Analyze the following meeting transcript and identify the main discussion topics.
    Return the topics as a clear bulleted list.
    
    Transcript:
    {state['transcript']}
    """
    response = llm.invoke([HumanMessage(content=prompt)])
    topics = [t.strip("- *") for t in response.content.strip().split("\n") if t.strip()]
    return {"topics": topics}

def summary_node(state: MeetingState):
    print("--- Running Meeting Summary Agent ---")
    prompt = f"""
    Generate a concise meeting summary explaining what the meeting was about. 
    The summary must be strictly limited to 3 to 5 sentences.
    
    Transcript:
    {state['transcript']}
    """
    response = llm.invoke([HumanMessage(content=prompt)])
    return {"summary": response.content}

def action_item_node(state: MeetingState):
    print("--- Running Action Item Extraction Agent ---")
    prompt = f"""
    Identify tasks assigned during the meeting and extract:
    - Task description
    - Person responsible (if mentioned). If the owner is not mentioned, label it as 'Owner: Not specified'.
    
    If there are NO action items, explicitly state "No action items identified".
    
    Transcript:
    {state['transcript']}
    """
    response = llm.invoke([HumanMessage(content=prompt)])
    content = response.content
    
    has_items = "no action items" not in content.lower()
    items = [i.strip("- *") for i in content.strip().split("\n") if i.strip()] if has_items else []
    
    return {
        "action_items": items,
        "has_action_items": has_items
    }

def priority_node(state: MeetingState):
    print("--- Running Priority Classification Agent ---")
    prompt = f"""
    Determine the priority level of the meeting tasks based on the transcript and action items.
    Possible outputs ONLY: 'High Priority', 'Medium Priority', or 'Low Priority'.
    Detect urgency from phrases like 'urgent', 'ASAP', 'by tomorrow', or 'this week'.
    
    Transcript:
    {state['transcript']}
    """
    response = llm.invoke([HumanMessage(content=prompt)])
    return {"priority": response.content.strip()}

def skip_priority_node(state: MeetingState):
    print("--- Skipping Priority Classification (No Action Items) ---")
    return {"priority": "No action items identified in this meeting."}

def final_output_node(state: MeetingState):
    print("--- Generating Final Structured Report ---")
    
    topics_str = "\n".join([f"- {t}" for t in state.get('topics', [])])
    actions_str = "\n".join([f"- {a}" for a in state.get('action_items', [])]) if state.get('has_action_items') else "No action items identified in this meeting."
    
    report = f"""
# 📋 Meeting Report

## 📝 Meeting Summary
{state.get('summary', 'N/A')}

## 🔑 Key Topics
{topics_str}

## ✅ Action Items & Owners
{actions_str}

## ⚡ Priority Level
{state.get('priority', 'N/A')}
    """
    return {"final_report": report}

In [ ]:
def check_action_items(state: MeetingState) -> Literal["priority_node", "skip_priority_node"]:
    if state.get("has_action_items", False):
        return "priority_node"
    else:
        return "skip_priority_node"

In [ ]:
workflow = StateGraph(MeetingState)

# Add nodes
workflow.add_node("topic_extraction", topic_extraction_node)
workflow.add_node("summary", summary_node)
workflow.add_node("action_items", action_item_node)
workflow.add_node("priority_node", priority_node)
workflow.add_node("skip_priority_node", skip_priority_node)
workflow.add_node("final_output", final_output_node)

# Set workflow path
workflow.add_edge(START, "topic_extraction")
workflow.add_edge("topic_extraction", "summary")
workflow.add_edge("summary", "action_items")

# Conditional routing branch
workflow.add_conditional_edges(
    "action_items",
    check_action_items,
    {
        "priority_node": "priority_node",
        "skip_priority_node": "skip_priority_node"
    }
)

# Merge paths back to final output
workflow.add_edge("priority_node", "final_output")
workflow.add_edge("skip_priority_node", "final_output")
workflow.add_edge("final_output", END)

# Compile graph
app = workflow.compile()
print("LangGraph workflow compiled successfully!")

In [ ]:
sample_transcript = """
John: We need to improve the website performance.
Sarah: Yes, page load time is too slow.
David: I will optimize the database queries this week.
Sarah: I will redesign the homepage layout.
John: Let's try to finish these tasks before Friday.
"""

initial_state = {
    "transcript": sample_transcript,
    "topics": [],
    "summary": "",
    "action_items": [],
    "has_action_items": False,
    "priority": "",
    "final_report": ""
}

result = app.invoke(initial_state)

print("\n" + "="*50)
print(result["final_report"])
print("="*50)